
# Ensemble / fusion детекций текста

Ноутбук предназначен для случая, когда **несколько моделей детекции** предсказали области текста для одного изображения, а на выходе нужна **одна итоговая разметка**.

Он сравнивает несколько стратегий:

1. **Score NMS** — классический baseline: среди пересекающихся предсказаний оставляем более уверенное.
2. **Soft-NMS** — не удаляет соседнее предсказание сразу, а уменьшает его score.
3. **Weighted Box Fusion (WBF)** — усредняет координаты близких bbox с весом по confidence.
4. **Consensus Best Polygon** — группирует совпавшие предсказания разных моделей и выбирает **реальную границу одной из моделей**, учитывая:
   - confidence;
   - нормализованный confidence модели;
   - число согласных моделей;
   - геометрическое согласие с другими предсказаниями.
5. **Consensus Medoid** — выбирает границу, которая геометрически наиболее близка ко всем остальным в группе.
6. **Strict Hybrid** — основной практический вариант: консенсус нескольких моделей + отсев слабых одиночных срабатываний.

Особенно важны последние три метода: они **не усредняют полигон в абстрактную форму**, а стараются выбрать одну наиболее правдоподобную исходную границу.

> Для текстовых строк обычный IoU иногда плох: одна модель может дать всю строку, а другая — только её часть. Поэтому кроме IoU используется `IoMin` (intersection / min(area1, area2)) и вертикальное совпадение строк.


In [ ]:

from pathlib import Path

RESULT_JSON = Path("result.json")
IMAGE_DIR = Path("images")
OUTPUT_DIR = Path("fusion_compare")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CFG = {
    "min_raw_score": 0.20,

    "raw_score_weight": 0.45,
    "rank_score_weight": 0.55,

    "cluster_affinity_threshold": 0.45,

    "nms_iou_threshold": 0.45,

    "soft_nms_sigma": 0.50,
    "soft_nms_min_score": 0.25,

    "consensus_min_models": 2,
    "strict_singleton_score": 0.86,

    "candidate_score_weight": 0.45,
    "candidate_support_weight": 0.30,
    "candidate_geometry_weight": 0.25,

    "provider_weights": {
        # "ppocr": 1.00,
        # "ppocr6": 1.00,
        # "rfdetr_historical": 1.05,
        # "docufcn": 1.05,
        # "eynollah_textline": 1.00,
        # "rtmdet_lines": 1.00,
    },
}

CFG


In [ ]:

import json
import math
import copy
from collections import defaultdict

import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt

with open(RESULT_JSON, "r", encoding="utf-8") as f:
    data = json.load(f)

print("image_name:", data.get("image_name"))
print("image_size:", data.get("image_size"))
print("providers:", list(data.get("providers", {}).keys()))
print("consensus_score from source:", data.get("consensus_score"))


In [ ]:

rows = []
for provider, block in data["providers"].items():
    anns = block.get("annotations", [])
    scores = [float(a.get("score", 0.0)) for a in anns]
    rows.append({
        "provider": provider,
        "model": block.get("model"),
        "detections": len(anns),
        "score_min": min(scores) if scores else np.nan,
        "score_mean": np.mean(scores) if scores else np.nan,
        "score_max": max(scores) if scores else np.nan,
        "elapsed_seconds": block.get("elapsed_seconds"),
    })

provider_stats = pd.DataFrame(rows).sort_values("provider")
display(provider_stats)

ax = provider_stats.set_index("provider")["score_mean"].plot(
    kind="bar", figsize=(10, 4), title="Средний raw score по моделям"
)
ax.set_ylabel("mean score")
plt.tight_layout()
plt.show()



## 1. Приведение детекций к общему виду

У разных детекторов могут отличаться:
- label (`text`, `text_line`);
- диапазоны confidence;
- форма polygon;
- наличие mask.

Для fusion используется общий объект:
- `bbox_xyxy`
- `polygon`
- `raw_score`
- `rank_score`
- `calibrated_score`
- `provider`

`rank_score` — percentile confidence **внутри данной модели**. Это делает сравнение разных моделей заметно честнее без размеченного validation set.

Если позже появится ground truth, лучше заменить эту эвристику на настоящую calibration: Platt / isotonic / temperature scaling.


In [ ]:

def xywh_to_xyxy(b):
    x, y, w, h = map(float, b)
    return [x, y, x + w, y + h]

def xyxy_to_xywh(b):
    x1, y1, x2, y2 = map(float, b)
    return [x1, y1, x2 - x1, y2 - y1]

def bbox_from_polygon(poly):
    arr = np.asarray(poly, dtype=float)
    return [
        float(arr[:, 0].min()),
        float(arr[:, 1].min()),
        float(arr[:, 0].max()),
        float(arr[:, 1].max()),
    ]

def rect_polygon_from_xyxy(b):
    x1, y1, x2, y2 = map(float, b)
    return [[x1, y1], [x2, y1], [x2, y2], [x1, y2]]

def provider_weight(name):
    return float(CFG["provider_weights"].get(name, 1.0))

def percentile_ranks(values):
    values = np.asarray(values, dtype=float)
    if len(values) <= 1:
        return np.ones_like(values)
    order = np.argsort(values)
    ranks = np.empty(len(values), dtype=float)
    ranks[order] = np.arange(len(values), dtype=float)
    return ranks / (len(values) - 1)

def flatten_predictions(data):
    detections = []

    for provider, block in data["providers"].items():
        anns = block.get("annotations", [])
        raw_scores = [float(a.get("score", 0.0)) for a in anns]
        ranks = percentile_ranks(raw_scores)

        for ann, rank in zip(anns, ranks):
            raw = float(ann.get("score", 0.0))
            if raw < CFG["min_raw_score"]:
                continue

            poly = ann.get("polygon")
            bbox_xywh = ann.get("bbox_xywh")

            if bbox_xywh is not None:
                bbox = xywh_to_xyxy(bbox_xywh)
            elif poly:
                bbox = bbox_from_polygon(poly)
            else:
                continue

            if not poly:
                poly = rect_polygon_from_xyxy(bbox)

            calibrated = (
                CFG["raw_score_weight"] * raw
                + CFG["rank_score_weight"] * float(rank)
            )
            calibrated *= provider_weight(provider)

            detections.append({
                "id": ann.get("id"),
                "provider": provider,
                "model": block.get("model"),
                "label": ann.get("label", "text"),
                "raw_score": raw,
                "rank_score": float(rank),
                "score": float(calibrated),
                "bbox": list(map(float, bbox)),
                "polygon": [[float(x), float(y)] for x, y in poly],
                "source_annotation": ann,
            })

    return detections

detections = flatten_predictions(data)
print("Всего детекций:", len(detections))

score_table = pd.DataFrame([
    {
        "provider": d["provider"],
        "raw_score": d["raw_score"],
        "rank_score": d["rank_score"],
        "calibrated_score": d["score"],
    }
    for d in detections
])

display(score_table.groupby("provider").agg(["count", "mean", "min", "max"]))


## 2. Геометрические метрики для текстовых строк

In [ ]:

def intersection_area(a, b):
    x1 = max(a[0], b[0])
    y1 = max(a[1], b[1])
    x2 = min(a[2], b[2])
    y2 = min(a[3], b[3])
    return max(0.0, x2 - x1) * max(0.0, y2 - y1)

def box_area(a):
    return max(0.0, a[2] - a[0]) * max(0.0, a[3] - a[1])

def iou(a, b):
    inter = intersection_area(a, b)
    union = box_area(a) + box_area(b) - inter
    return inter / union if union > 0 else 0.0

def io_min(a, b):
    inter = intersection_area(a, b)
    denom = min(box_area(a), box_area(b))
    return inter / denom if denom > 0 else 0.0

def axis_overlap_ratio(a1, a2, b1, b2):
    inter = max(0.0, min(a2, b2) - max(a1, b1))
    denom = min(max(1e-9, a2-a1), max(1e-9, b2-b1))
    return inter / denom

def line_affinity(a, b):
    # Сходство именно для text-line detection.
    # IoMin позволяет сопоставить длинную строку с коротким фрагментом.
    A, B = a["bbox"], b["bbox"]
    i = iou(A, B)
    iom = io_min(A, B)
    yov = axis_overlap_ratio(A[1], A[3], B[1], B[3])

    h1 = max(1.0, A[3] - A[1])
    h2 = max(1.0, B[3] - B[1])
    cy1 = (A[1] + A[3]) / 2
    cy2 = (B[1] + B[3]) / 2
    center_y_sim = math.exp(-abs(cy1 - cy2) / max(h1, h2))

    return (
        0.25 * i
        + 0.35 * iom
        + 0.25 * yov
        + 0.15 * center_y_sim
    )

def cluster_detections(dets, threshold=None):
    if threshold is None:
        threshold = CFG["cluster_affinity_threshold"]

    remaining = sorted(dets, key=lambda d: d["score"], reverse=True)
    clusters = []

    while remaining:
        seed = remaining.pop(0)
        cluster = [seed]
        changed = True

        while changed:
            changed = False
            keep = []
            for d in remaining:
                best = max(line_affinity(d, c) for c in cluster)
                if best >= threshold:
                    cluster.append(d)
                    changed = True
                else:
                    keep.append(d)
            remaining = keep

        clusters.append(cluster)

    return clusters

clusters = cluster_detections(detections)
print("Кластеров:", len(clusters))
print("Размеры 20 крупнейших:", sorted([len(c) for c in clusters], reverse=True)[:20])


## 3. Метод 1 — обычный Score NMS

In [ ]:

def score_nms(dets, iou_threshold=None):
    if iou_threshold is None:
        iou_threshold = CFG["nms_iou_threshold"]

    work = [copy.deepcopy(d) for d in dets]
    work.sort(key=lambda d: d["score"], reverse=True)
    kept = []

    while work:
        best = work.pop(0)
        kept.append(best)
        work = [d for d in work if iou(best["bbox"], d["bbox"]) < iou_threshold]

    return kept


## 4. Метод 2 — Soft-NMS

In [ ]:

def soft_nms(dets, sigma=None, min_score=None):
    if sigma is None:
        sigma = CFG["soft_nms_sigma"]
    if min_score is None:
        min_score = CFG["soft_nms_min_score"]

    work = [copy.deepcopy(d) for d in dets]
    out = []

    while work:
        work.sort(key=lambda d: d["score"], reverse=True)
        best = work.pop(0)
        out.append(best)

        next_work = []
        for d in work:
            ov = iou(best["bbox"], d["bbox"])
            d["score"] *= math.exp(-(ov * ov) / sigma)
            if d["score"] >= min_score:
                next_work.append(d)

        work = next_work

    return out


## 5. Метод 3 — Weighted Box Fusion (bbox)

In [ ]:

def weighted_box_fusion(dets):
    clusters = cluster_detections(dets)
    fused = []

    for idx, cl in enumerate(clusters):
        weights = np.asarray(
            [max(1e-6, d["score"]) * provider_weight(d["provider"]) for d in cl],
            dtype=float,
        )
        boxes = np.asarray([d["bbox"] for d in cl], dtype=float)
        fused_box = np.average(boxes, axis=0, weights=weights)

        providers = sorted(set(d["provider"] for d in cl))
        max_score = max(d["score"] for d in cl)
        mean_raw = float(np.average([d["raw_score"] for d in cl], weights=weights))

        fused.append({
            "id": f"wbf-{idx:04d}",
            "provider": "fusion",
            "model": "Weighted Box Fusion",
            "label": "text_line",
            "raw_score": mean_raw,
            "rank_score": max(d["rank_score"] for d in cl),
            "score": float(max_score),
            "bbox": fused_box.tolist(),
            "polygon": rect_polygon_from_xyxy(fused_box.tolist()),
            "support_models": len(providers),
            "support_providers": providers,
            "cluster_size": len(cl),
        })

    return fused



## 6. Метод 4 — Consensus Best Polygon

Для каждого кластера оценивается **каждая исходная граница**.

Итоговая оценка складывается из:
- calibrated score конкретной детекции;
- числа моделей, которые поддерживают эту область;
- геометрического согласия с остальными предсказаниями.

В отличие от WBF здесь сохраняется **реальный полигон одной из моделей**.


In [ ]:

def cluster_support(cl, total_providers):
    return len(set(d["provider"] for d in cl)) / max(1, total_providers)

def candidate_geometry_score(candidate, cl):
    others = [d for d in cl if d is not candidate]
    if not others:
        return 0.0

    vals = [
        line_affinity(candidate, other) * max(0.05, other["score"])
        for other in others
    ]
    weights = [max(0.05, other["score"]) for other in others]
    return float(np.average(vals, weights=weights))

def consensus_best_polygon(dets):
    clusters = cluster_detections(dets)
    total_providers = len(set(d["provider"] for d in dets))
    out = []

    for idx, cl in enumerate(clusters):
        support = cluster_support(cl, total_providers)
        support_providers = sorted(set(d["provider"] for d in cl))

        candidates = []
        for d in cl:
            geom = candidate_geometry_score(d, cl)
            quality = (
                CFG["candidate_score_weight"] * d["score"]
                + CFG["candidate_support_weight"] * support
                + CFG["candidate_geometry_weight"] * geom
            )
            candidates.append((quality, geom, d))

        quality, geom, best = max(candidates, key=lambda x: x[0])
        item = copy.deepcopy(best)
        item.update({
            "id": f"consensus-best-{idx:04d}",
            "fusion_method": "consensus_best_polygon",
            "final_quality": float(quality),
            "geometry_score": float(geom),
            "support_models": len(support_providers),
            "support_providers": support_providers,
            "cluster_size": len(cl),
        })
        out.append(item)

    return out


## 7. Метод 5 — Consensus Medoid

In [ ]:

def consensus_medoid(dets):
    clusters = cluster_detections(dets)
    out = []

    for idx, cl in enumerate(clusters):
        providers = sorted(set(d["provider"] for d in cl))
        scored = []

        for candidate in cl:
            if len(cl) == 1:
                geom = 0.0
            else:
                similarities = []
                weights = []

                for other in cl:
                    if other is candidate:
                        continue
                    similarities.append(line_affinity(candidate, other))
                    weights.append(max(0.05, other["score"]))

                geom = float(np.average(similarities, weights=weights))

            medoid_score = 0.72 * geom + 0.28 * candidate["score"]
            scored.append((medoid_score, geom, candidate))

        medoid_score, geom, best = max(scored, key=lambda x: x[0])

        item = copy.deepcopy(best)
        item.update({
            "id": f"medoid-{idx:04d}",
            "fusion_method": "consensus_medoid",
            "final_quality": float(medoid_score),
            "geometry_score": float(geom),
            "support_models": len(providers),
            "support_providers": providers,
            "cluster_size": len(cl),
        })
        out.append(item)

    return out



## 8. Метод 6 — Strict Hybrid

Практический вариант для получения более чистой единственной разметки:

- если область подтверждена несколькими моделями — оставляем;
- если область нашла только одна модель — оставляем её только при очень высоком score.

Такой режим обычно уменьшает число ложных областей.


In [ ]:

def strict_hybrid(dets):
    fused = consensus_best_polygon(dets)
    out = []

    for d in fused:
        support = int(d.get("support_models", 1))

        if support >= CFG["consensus_min_models"]:
            out.append(d)
        elif d["score"] >= CFG["strict_singleton_score"]:
            out.append(d)

    return out


## 9. Запуск всех методов

In [ ]:

methods = {
    "01_score_nms": lambda: score_nms(detections),
    "02_soft_nms": lambda: soft_nms(detections),
    "03_wbf": lambda: weighted_box_fusion(detections),
    "04_consensus_best_polygon": lambda: consensus_best_polygon(detections),
    "05_consensus_medoid": lambda: consensus_medoid(detections),
    "06_strict_hybrid": lambda: strict_hybrid(detections),
}

results = {name: fn() for name, fn in methods.items()}

summary = pd.DataFrame([
    {
        "method": name,
        "detections": len(preds),
        "mean_score": np.mean([p["score"] for p in preds]) if preds else np.nan,
        "mean_support": np.mean([p.get("support_models", 1) for p in preds]) if preds else np.nan,
    }
    for name, preds in results.items()
])

display(summary)


## 10. Визуализация и экспорт папок

In [ ]:

def load_base_image(data, image_dir):
    image_name = data.get("image_name")
    image_path = image_dir / image_name if image_name else None

    if image_path and image_path.exists():
        return Image.open(image_path).convert("RGB"), image_path

    width, height = data.get("image_size", [1440, 1172])
    return Image.new("RGB", (int(width), int(height)), "white"), None

def draw_predictions(base_img, preds, show_labels=True, width=3):
    img = base_img.copy()
    draw = ImageDraw.Draw(img)

    for i, d in enumerate(preds):
        poly = [(float(x), float(y)) for x, y in d["polygon"]]

        if len(poly) >= 2:
            draw.line(poly + [poly[0]], fill=(255, 0, 0), width=width)

        if show_labels and poly:
            score = d.get("score", 0.0)
            support = d.get("support_models", 1)
            provider = d.get("provider", "?")
            txt = f"{i} {provider} s={score:.2f} m={support}"
            x, y = poly[0]
            draw.text((x + 2, max(0, y - 12)), txt, fill=(0, 80, 255))

    return img

def export_method_json(method_name, preds, out_dir):
    payload = {
        "run_id": data.get("run_id"),
        "image_name": data.get("image_name"),
        "image_size": data.get("image_size"),
        "fusion_method": method_name,
        "config": CFG,
        "annotations": [],
    }

    for i, d in enumerate(preds):
        payload["annotations"].append({
            "id": d.get("id", f"{method_name}-{i:04d}"),
            "label": d.get("label", "text_line"),
            "score": float(d.get("score", 0.0)),
            "raw_score": float(d.get("raw_score", 0.0)),
            "bbox_xywh": xyxy_to_xywh(d["bbox"]),
            "polygon": d["polygon"],
            "selected_from_provider": d.get("provider"),
            "support_models": int(d.get("support_models", 1)),
            "support_providers": d.get("support_providers", [d.get("provider")]),
            "geometry_score": d.get("geometry_score"),
            "final_quality": d.get("final_quality"),
        })

    with open(out_dir / "result.json", "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

base_img, source_image_path = load_base_image(data, IMAGE_DIR)
print("Исходное изображение:", source_image_path or "НЕ НАЙДЕНО — используется белый холст")

for method_name, preds in results.items():
    method_dir = OUTPUT_DIR / method_name
    method_dir.mkdir(parents=True, exist_ok=True)

    vis = draw_predictions(base_img, preds)
    vis.save(method_dir / data.get("image_name", "preview.png"))
    export_method_json(method_name, preds, method_dir)

summary.to_csv(OUTPUT_DIR / "summary.csv", index=False)

print("Готово:", OUTPUT_DIR.resolve())


## 11. Сравнение всех методов

In [ ]:

for method_name, preds in results.items():
    vis = draw_predictions(base_img, preds, show_labels=False, width=2)

    plt.figure(figsize=(14, 10))
    plt.imshow(vis)
    plt.title(f"{method_name} | detections={len(preds)}")
    plt.axis("off")
    plt.show()



## 12. Диагностика Consensus

Полезные признаки:

- `support_models = 1` — область нашла только одна модель;
- `support_models >= 2` — есть межмодельное подтверждение;
- высокий `geometry_score` — модели хорошо согласны по положению границы;
- высокий `score`, но `support_models = 1` — либо ценная уникальная находка, либо false positive.


In [ ]:

cons = pd.DataFrame([
    {
        "id": d.get("id"),
        "provider": d.get("provider"),
        "score": d.get("score"),
        "raw_score": d.get("raw_score"),
        "support_models": d.get("support_models", 1),
        "support_providers": ", ".join(d.get("support_providers", [])),
        "geometry_score": d.get("geometry_score"),
        "final_quality": d.get("final_quality"),
        "x1": d["bbox"][0],
        "y1": d["bbox"][1],
        "x2": d["bbox"][2],
        "y2": d["bbox"][3],
    }
    for d in results["04_consensus_best_polygon"]
])

cons = cons.sort_values(
    ["support_models", "final_quality"],
    ascending=[False, False],
)

display(cons.head(100))
cons.to_csv(OUTPUT_DIR / "consensus_diagnostics.csv", index=False)



## 13. Что смотреть в первую очередь

Для вашей задачи сначала сравните:

### `04_consensus_best_polygon`
Баланс между полнотой и качеством границы. Сохраняет полигон конкретной модели.

### `05_consensus_medoid`
Полезен, если raw confidence моделей плохо сопоставим и хочется сильнее опираться на геометрическое согласие.

### `06_strict_hybrid`
Подходит, если главный приоритет — убрать ложные области и получить максимально чистую итоговую разметку.

Параметры для первых экспериментов:

```python
CFG["cluster_affinity_threshold"] = 0.40   # больше объединений
CFG["cluster_affinity_threshold"] = 0.50   # строже

CFG["strict_singleton_score"] = 0.80       # больше recall
CFG["strict_singleton_score"] = 0.90       # меньше false positive
```

Если появится размеченный validation set, параметры можно подбирать автоматически по polygon IoU / Dice / precision / recall / F1.
